In [ ]:
%sql
-- ============================================================
-- Agentic Restock Workflow — §4.2 Deep Analysis, as Unity Catalog functions
-- ============================================================
-- Registers the Genie Agent's deep-analysis logic (consumption trend,
-- stockout forecast, urgency classification, quote line-item math, veto)
-- as governed Unity Catalog SQL functions in ab_training.agentic_restock.
--
-- Why UC functions instead of a notebook/job task: per Databricks Agent
-- Bricks docs, this is the correct primitive for "complex logic that
-- cannot be captured with a static or parameterized SQL query" — they can
-- be registered as trusted SQL functions on a Genie Agent, added directly
-- as tools on a Supervisor Agent, and queried by anyone with EXECUTE
-- permission, all without duplicating the logic in Python.
-- ============================================================

In [ ]:
%sql
-- ============================================================
-- Function 1: avg_daily_consumption
-- Trailing-window average daily consumption, anchored to today
-- (current_date()). In production, consumption_history is refreshed
-- continuously so "today - lookback_days" is exactly the trailing
-- window. NOTE for this mock dataset: rows are seeded for a fixed
-- 2026-08-01..2026-08-14 range, so as real time moves past that
-- range the window naturally covers fewer seed days — expected
-- behavior for static mock data, not a bug. Kept as a single flat
-- query (no subqueries/window functions) because Databricks SQL
-- functions reject correlated subqueries once another function
-- calls this one and its body gets inlined.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.avg_daily_consumption(
  item_id STRING COMMENT 'Part SKU identifier, e.g. PRT-BRK-001',
  warehouse_id STRING COMMENT 'Warehouse code, e.g. WH-BLR-01',
  lookback_days INT DEFAULT 14 COMMENT 'Trailing window size in days, ending today'
)
RETURNS DOUBLE
COMMENT 'Average daily consumption (architecture §4.2) over the trailing `lookback_days` ending today. Returns 0.0 if no consumption history exists in that window.'
RETURN
  SELECT COALESCE(AVG(qty_consumed), 0.0)
  FROM ab_training.agentic_restock.consumption_history
  WHERE item_id = avg_daily_consumption.item_id
    AND warehouse_id = avg_daily_consumption.warehouse_id
    AND consumption_date > date_sub(current_date(), avg_daily_consumption.lookback_days);

In [ ]:
%sql
-- ============================================================
-- Function 2: predicted_stockout_date
-- Forward-looking forecast from *today* (real time), using the
-- current on-hand stock and the trailing avg_daily_consumption.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.predicted_stockout_date(
  item_id STRING COMMENT 'Part SKU identifier',
  warehouse_id STRING COMMENT 'Warehouse code'
)
RETURNS DATE
COMMENT 'Earliest predicted stockout date (architecture §4.2), projected from today using current_stock_qty and the trailing 14-day avg_daily_consumption. NULL when consumption is ~0 (nothing to forecast).'
RETURN
  -- MAX(...) wrappers are no-ops (exactly one row matches the PK filter) but
  -- are required: Databricks SQL functions reject a table-scanning body
  -- whose SELECT list isn't provably single-row via aggregation once the
  -- function is called from inside another function's body — a PK-filtered
  -- WHERE alone isn't accepted as proof.
  SELECT
    CASE
      WHEN MAX(ab_training.agentic_restock.avg_daily_consumption(isl.item_id, isl.warehouse_id, 14)) > 0
      THEN date_add(
        current_date(),
        CAST(CEIL(
          MAX(isl.current_stock_qty)
          / MAX(ab_training.agentic_restock.avg_daily_consumption(isl.item_id, isl.warehouse_id, 14))
        ) AS INT)
      )
      ELSE NULL
    END
  FROM ab_training.agentic_restock.inventory_stock_level isl
  WHERE isl.item_id = predicted_stockout_date.item_id
    AND isl.warehouse_id = predicted_stockout_date.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 3: classify_urgency
-- Pure classification, no table access — an absolute floor breach
-- (<= minimum_stock_qty) is always CRITICAL regardless of forecast;
-- otherwise urgency is banded by days_remaining until stockout.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.classify_urgency(
  current_stock_qty INT COMMENT 'Current stock on hand',
  minimum_stock_qty INT COMMENT 'Absolute floor for CRITICAL scoring',
  days_remaining DOUBLE COMMENT 'Days until predicted stockout, or NULL if no forecast (near-zero consumption)'
)
RETURNS STRING
COMMENT 'Urgency classification per architecture §4.2: CRITICAL (at/below minimum_stock_qty, or <=3 days to stockout), HIGH (<=7 days), MEDIUM (<=14 days), LOW (>14 days or no forecastable consumption).'
RETURN
  CASE
    WHEN classify_urgency.current_stock_qty <= classify_urgency.minimum_stock_qty THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining IS NULL THEN 'LOW'
    WHEN classify_urgency.days_remaining <= 3 THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining <= 7 THEN 'HIGH'
    WHEN classify_urgency.days_remaining <= 14 THEN 'MEDIUM'
    ELSE 'LOW'
  END;

In [ ]:
%sql
-- ============================================================
-- Function 4: requested_restock_qty
-- Quote line-item math: how many units to order to reach target
-- stock. Floored at 0 (never a negative order).
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.requested_restock_qty(
  item_id STRING COMMENT 'Part SKU identifier',
  warehouse_id STRING COMMENT 'Warehouse code'
)
RETURNS INT
COMMENT 'Suggested restock quantity: target_stock_qty - current_stock_qty, floored at 0. NULL if the item/warehouse has no active threshold config.'
RETURN
  SELECT MAX(GREATEST(tct.target_stock_qty - isl.current_stock_qty, 0))
  FROM ab_training.agentic_restock.inventory_stock_level isl
  JOIN ab_training.agentic_restock.threshold_config_table tct
    ON isl.item_id = tct.item_id AND isl.warehouse_id = tct.warehouse_id
  WHERE isl.item_id = requested_restock_qty.item_id
    AND isl.warehouse_id = requested_restock_qty.warehouse_id
    AND tct.is_active = true;

In [ ]:
%sql
-- ============================================================
-- Function 5: needs_restock (veto power)
-- Genie Agent's veto: is this Lakeflow-flagged candidate a false
-- positive? STUB — always TRUE today. There is no inbound-shipment
-- or supply-chain table yet, so there is nothing to veto against.
-- Once that data source exists, rewrite the RETURN body to check
-- for an inbound shipment covering the shortfall and return FALSE
-- in that case.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.needs_restock(
  item_id STRING COMMENT 'Part SKU identifier',
  warehouse_id STRING COMMENT 'Warehouse code'
)
RETURNS BOOLEAN
COMMENT 'Veto decision (architecture §4.2): whether this candidate genuinely needs restocking, or is a false positive (e.g. an inbound shipment already covers the shortfall). STUB: always TRUE until an inbound-shipment data source exists.'
RETURN TRUE;

In [ ]:
%sql
-- ============================================================
-- Function 6: restock_candidate_summary
-- Deterministic natural-language one-liner combining functions
-- 1-4. This is the function most useful as a Genie Agent trusted
-- asset (answers "why does X need restocking?") and as the text
-- source for the Teams Adaptive Card / open_request.summary_report.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.restock_candidate_summary(
  item_id STRING COMMENT 'Part SKU identifier',
  warehouse_id STRING COMMENT 'Warehouse code'
)
RETURNS STRING
COMMENT 'Deterministic natural-language summary of one restock candidate: stock on hand, avg daily consumption, predicted stockout date, urgency, and suggested reorder quantity (architecture §4.2).'
RETURN
  SELECT MAX(CONCAT(
    isl.item_name, ' at ', isl.warehouse_id, ': ',
    CAST(isl.current_stock_qty AS STRING), ' ', isl.unit_of_measure, ' on hand (reorder point ',
    CAST(tct.reorder_point_qty AS STRING), '). Avg consumption ',
    CAST(ROUND(ab_training.agentic_restock.avg_daily_consumption(isl.item_id, isl.warehouse_id, 14), 1) AS STRING),
    '/day. ',
    CASE
      WHEN ab_training.agentic_restock.predicted_stockout_date(isl.item_id, isl.warehouse_id) IS NOT NULL
      THEN CONCAT('Predicted stockout ', CAST(ab_training.agentic_restock.predicted_stockout_date(isl.item_id, isl.warehouse_id) AS STRING), '. ')
      ELSE 'No forecastable stockout (near-zero consumption). '
    END,
    'Urgency: ', ab_training.agentic_restock.classify_urgency(
      isl.current_stock_qty,
      tct.minimum_stock_qty,
      datediff(ab_training.agentic_restock.predicted_stockout_date(isl.item_id, isl.warehouse_id), current_date())
    ), '. ',
    'Suggested reorder: ', CAST(ab_training.agentic_restock.requested_restock_qty(isl.item_id, isl.warehouse_id) AS STRING), ' ', isl.unit_of_measure, '.'
  ))
  FROM ab_training.agentic_restock.inventory_stock_level isl
  JOIN ab_training.agentic_restock.threshold_config_table tct
    ON isl.item_id = tct.item_id AND isl.warehouse_id = tct.warehouse_id
  WHERE isl.item_id = restock_candidate_summary.item_id
    AND isl.warehouse_id = restock_candidate_summary.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Verification: run all 6 functions against every candidate the
-- §4.1 coarse check would flag (current_stock_qty <= reorder_point_qty).
-- Eyeball these against the mock data in schema_bootstrap.ipynb.
-- ============================================================

SELECT
  isl.item_id,
  isl.item_name,
  isl.warehouse_id,
  isl.current_stock_qty,
  tct.reorder_point_qty,
  tct.minimum_stock_qty,
  ROUND(ab_training.agentic_restock.avg_daily_consumption(isl.item_id, isl.warehouse_id, 14), 2) AS avg_daily_consumption,
  ab_training.agentic_restock.predicted_stockout_date(isl.item_id, isl.warehouse_id) AS predicted_stockout_date,
  ab_training.agentic_restock.classify_urgency(
    isl.current_stock_qty,
    tct.minimum_stock_qty,
    datediff(ab_training.agentic_restock.predicted_stockout_date(isl.item_id, isl.warehouse_id), current_date())
  ) AS urgency_level,
  ab_training.agentic_restock.requested_restock_qty(isl.item_id, isl.warehouse_id) AS requested_restock_qty,
  ab_training.agentic_restock.needs_restock(isl.item_id, isl.warehouse_id) AS needs_restock,
  ab_training.agentic_restock.restock_candidate_summary(isl.item_id, isl.warehouse_id) AS summary
FROM ab_training.agentic_restock.inventory_stock_level isl
JOIN ab_training.agentic_restock.threshold_config_table tct
  ON isl.item_id = tct.item_id AND isl.warehouse_id = tct.warehouse_id
WHERE tct.is_active = true
  AND isl.current_stock_qty <= tct.reorder_point_qty
ORDER BY urgency_level, isl.item_id;